In [34]:
# %pip install polars
# %pip install duckdb
# %pip install pyarrow

In [1]:
import polars as pl
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import numpy as np
import duckdb
import warnings
import pyarrow

# Suprimir os warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

## Base Sisvan - Estado Nutricional

##### Convertendo arquivo .csv para .parquet para reduzir o tamanho do arquivo

In [2]:
def read_csv(file_path):
    duckdb.execute(f"""
        COPY (
            SELECT * FROM read_csv(
                '{file_path}',
                delim=';',
                header=true,
                encoding='latin-1',
                strict_mode=false,
                ignore_errors=true,
                quote='"'
            )
        )
        TO 'sisvan_estado_nutricional_2023.parquet'
        (FORMAT PARQUET, COMPRESSION SNAPPY)
    """)

read_csv('sisvan_estado_nutricional_2023.csv')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
df = pl.read_parquet('sisvan_estado_nutricional_2023.parquet')
df.shape

(53981528, 34)

In [4]:
df.schema

Schema([('CO_ACOMPANHAMENTO', String),
        ('CO_PESSOA_SISVAN', String),
        ('ST_PARTICIPA_ANDI', String),
        ('CO_MUNICIPIO_IBGE', Int64),
        ('SG_UF', String),
        ('NO_MUNICIPIO', String),
        ('CO_CNES', String),
        ('NU_IDADE_ANO', Int64),
        ('NU_FASE_VIDA', Float64),
        ('DS_FASE_VIDA', String),
        ('SG_SEXO', String),
        ('CO_RACA_COR', String),
        ('DS_RACA_COR', String),
        ('CO_POVO_COMUNIDADE', Int64),
        ('DS_POVO_COMUNIDADE', String),
        ('CO_ESCOLARIDADE', String),
        ('DS_ESCOLARIDADE', String),
        ('DT_ACOMPANHAMENTO', Date),
        ('NU_COMPETENCIA', Int64),
        ('NU_PESO', String),
        ('NU_ALTURA', String),
        ('DS_IMC', String),
        ('DS_IMC_PRE_GESTACIONAL', Float64),
        ('PESO X IDADE', String),
        ('PESO X ALTURA', String),
        ('CRI. ALTURA X IDADE', String),
        ('CRI. IMC X IDADE', String),
        ('ADO. ALTURA X IDADE', String),
        ('AD

In [22]:
df.dtypes

[String,
 String,
 String,
 Int64,
 String,
 String,
 String,
 Int64,
 Float64,
 String,
 String,
 String,
 String,
 Int64,
 String,
 String,
 String,
 Date,
 Int64,
 String,
 String,
 String,
 Float64,
 String,
 String,
 String,
 String,
 String,
 String,
 String,
 String,
 String,
 Int64,
 String]

In [19]:
df.head()

CO_ACOMPANHAMENTO,CO_PESSOA_SISVAN,ST_PARTICIPA_ANDI,CO_MUNICIPIO_IBGE,SG_UF,NO_MUNICIPIO,CO_CNES,NU_IDADE_ANO,NU_FASE_VIDA,DS_FASE_VIDA,SG_SEXO,CO_RACA_COR,DS_RACA_COR,CO_POVO_COMUNIDADE,DS_POVO_COMUNIDADE,CO_ESCOLARIDADE,DS_ESCOLARIDADE,DT_ACOMPANHAMENTO,NU_COMPETENCIA,NU_PESO,NU_ALTURA,DS_IMC,DS_IMC_PRE_GESTACIONAL,PESO X IDADE,PESO X ALTURA,CRI. ALTURA X IDADE,CRI. IMC X IDADE,ADO. ALTURA X IDADE,ADO. IMC X IDADE,CO_ESTADO_NUTRI_ADULTO,CO_ESTADO_NUTRI_IDOSO,CO_ESTADO_NUTRI_IMC_SEMGEST,CO_SISTEMA_ORIGEM_ACOMP,SISTEMA_ORIGEM_ACOMP
str,str,str,i64,str,str,str,i64,f64,str,str,str,str,i64,str,str,str,date,i64,str,str,str,f64,str,str,str,str,str,str,str,str,str,i64,str
"""E0806740A9F53A4B389C44D87FA484…","""22FD58637A3250A35D8A6664ECDFA6…",null,350760,"""SP""","""BRAGANCA PAULISTA""","""2688093""",65,8.0,"""IDOSO""","""M""","""01""","""BRANCA""",null,"""NÃO INFORMADO""",null,"""SEM INFORMAÇÃO""",2023-01-30,202301,"""67""","""175""","""21,88""",null,null,null,null,null,null,null,null,"""Baixo peso""",null,4,"""E-SUS AB"""
"""E296774CC772FD6D6606AE581F7DDA…","""E5F2FC1AB9536BEDD3C99CBB83394B…",null,130063,"""AM""","""BERURI""","""0843296""",29,7.0,"""ADULTO""","""M""","""03""","""AMARELA""",null,"""NÃO INFORMADO""",null,"""SEM INFORMAÇÃO""",2023-01-17,202301,"""120,2""","""172""","""40,63""",null,null,null,null,null,null,null,"""Obesidade Grau III""",null,null,4,"""E-SUS AB"""
"""D80817E95FAFA950EEB3E0A5B44FD2…","""30DCEACC2CD77956C6E4E16FA1D81D…",null,315050,"""MG""","""PIMENTA""","""3912175""",15,6.0,"""ADOLESCENTE""","""M""","""01""","""BRANCA""",null,"""NÃO INFORMADO""","""99""","""SEM INFORMAÇÃO""",2023-01-19,202301,"""57""","""171""","""19,49""",null,null,null,null,null,"""Estatura adequada para a idade""","""Eutrofia""",null,null,null,4,"""E-SUS AB"""
"""54E5B26F06555AF1C798C323B4FD84…","""648B3D013A6024BEB84CA05BD9130F…",null,500020,"""MS""","""AGUA CLARA""","""3989372""",58,7.0,"""ADULTO""","""M""","""04""","""PARDA""",null,"""NÃO INFORMADO""",null,"""SEM INFORMAÇÃO""",2023-01-30,202301,"""100""","""172""","""33,8""",null,null,null,null,null,null,null,"""Obesidade Grau I""",null,null,4,"""E-SUS AB"""
"""64B0C6D9F18FC845BDE7CCF79B9C4F…","""A8244673295286A331F73DC786BFB4…",null,260650,"""PE""","""IATI""","""2712350""",14,6.0,"""ADOLESCENTE""","""F""","""03""","""AMARELA""",null,"""NÃO INFORMADO""","""99""","""SEM INFORMAÇÃO""",2023-01-11,202301,"""43,4""","""153""","""18,54""",null,null,null,null,null,"""Estatura adequada para a idade""","""Eutrofia""",null,null,null,4,"""E-SUS AB"""


In [37]:
null_counts = df.null_count().row(0)
for col, n in zip(df.columns, null_counts):
    print(f"{col}: {n}")

CO_ACOMPANHAMENTO: 0
CO_PESSOA_SISVAN: 0
ST_PARTICIPA_ANDI: 53981528
CO_MUNICIPIO_IBGE: 0
SG_UF: 0
NO_MUNICIPIO: 0
CO_CNES: 225681
NU_IDADE_ANO: 0
NU_FASE_VIDA: 0
DS_FASE_VIDA: 0
SG_SEXO: 0
CO_RACA_COR: 0
DS_RACA_COR: 0
CO_POVO_COMUNIDADE: 53811679
DS_POVO_COMUNIDADE: 0
CO_ESCOLARIDADE: 39938896
DS_ESCOLARIDADE: 0
DT_ACOMPANHAMENTO: 0
NU_COMPETENCIA: 0
NU_PESO: 0
NU_ALTURA: 0
DS_IMC: 0
DS_IMC_PRE_GESTACIONAL: 53975922
PESO X IDADE: 40742037
PESO X ALTURA: 46525813
CRI. ALTURA X IDADE: 40742787
CRI. IMC X IDADE: 40761613
ADO. ALTURA X IDADE: 46640236
ADO. IMC X IDADE: 46649063
CO_ESTADO_NUTRI_ADULTO: 29198499
CO_ESTADO_NUTRI_IDOSO: 45690997
CO_ESTADO_NUTRI_IMC_SEMGEST: 53237992
CO_SISTEMA_ORIGEM_ACOMP: 0
SISTEMA_ORIGEM_ACOMP: 0


In [ ]:
crianca = df.filter(pl.col("CRI. IMC X IDADE").is_not_null())
adolescente = df.filter(pl.col("ADO. IMC X IDADE").is_not_null())
adulto = df.filter(
    pl.col("CO_ESTADO_NUTRI_ADULTO").is_not_null()
)
idoso = df.filter(
    pl.col("CO_ESTADO_NUTRI_IDOSO").is_not_null()
)

In [39]:
adulto.shape

(53981528, 34)

In [40]:
adolescente.shape

(7332465, 34)

In [41]:
crianca.shape

(13219915, 34)

In [ ]:
idoso.shape